In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# מעברי Transpiler מבוססי בינה מלאכותית

מעברי ה-Transpiler המבוססים על בינה מלאכותית הם מעברים שמשמשים כתחליף ישיר למעברי Qiskit "המסורתיים" עבור משימות transpiling מסוימות. הם לרוב מניבים תוצאות טובות יותר מאלגוריתמים היוריסטיים קיימים (כמו עומק נמוך יותר ומספר שערי CNOT קטן יותר), אך גם מהירים בהרבה מאלגוריתמי אופטימיזציה כמו פותרי Boolean satisfiability. מעברי ה-AI Transpiler רצים בסביבה המקומית שלך.

> **Note:** מעברי ה-Transpiler המבוססים על בינה מלאכותית נמצאים בסטטוס שחרור בטא, וכפופים לשינויים.
> אם יש לך משוב או שאתה רוצה ליצור קשר עם צוות המפתחים, השתמש ב[ערוץ Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) הזה.

המעברים הבאים זמינים כרגע:

**מעברי Routing**
 - `AIRouting`: בחירת פריסה (Layout) וניתוב מעגלים (Routing)

**מעברי סינתזת מעגלים**
 - `AICliffordSynthesis`: סינתזה של מעגלי Clifford
 - `AILinearFunctionSynthesis`: סינתזה של מעגלי Linear Function
 - `AIPermutationSynthesis`: סינתזה של מעגלי Permutation
כדי להשתמש במעברי ה-AI Transpiler, קודם התקן את חבילת `qiskit-ibm-transpiler`. כנס ל[תיעוד ה-API של qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) לקבלת מידע נוסף על האפשרויות השונות הזמינות.

## הרץ את מעברי ה-AI Transpiler מקומית או בענן
קודם התקן את `qiskit-ibm-transpiler` עם כמה תלויות נוספות כך:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

לאחר התקנת התלויות הנוספות, מצב ברירת המחדל להרצת מעברי ה-Transpiler המבוססים על בינה מלאכותית הוא שימוש במחשב המקומי שלך.

## מעבר AI Routing
המעבר `AIRouting` משמש גם כשלב פריסה (layout) וגם כשלב ניתוב (routing). ניתן להשתמש בו בתוך `PassManager` כך:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

כאן, ה-`backend` קובע עבור איזו מפת צימוד (coupling map) לנתב, ה-`optimization_level` (1, 2, או 3) קובע את מאמץ החישוב שיושקע בתהליך (ערך גבוה יותר בדרך כלל נותן תוצאות טובות יותר אך לוקח יותר זמן), וה-`layout_mode` מציין כיצד לטפל בבחירת הפריסה.
ל-`layout_mode` יש את האפשרויות הבאות:

- `keep`: זה מכבד את הפריסה שנקבעה על ידי מעברי ה-Transpiler הקודמים (או משתמש בפריסה הטריוויאלית אם לא נקבעה). בדרך כלל משתמשים בו רק כאשר המעגל חייב לרוץ על qubits ספציפיים של המכשיר. לרוב מניב תוצאות גרועות יותר מכיוון שיש לו פחות מרחב לאופטימיזציה.
- `improve`: זה משתמש בפריסה שנקבעה על ידי מעברי ה-Transpiler הקודמים כנקודת התחלה. שימושי כשיש לך ניחוש ראשוני טוב לפריסה; לדוגמה, עבור מעגלים שנבנו בצורה שמשקפת בערך את מפת הצימוד של המכשיר. שימושי גם אם אתה רוצה לנסות מעברי פריסה ספציפיים אחרים בשילוב עם מעבר `AIRouting`.
- `optimize`: זהו המצב הברירת מחדל. עובד הכי טוב עבור מעגלים כלליים שאולי אין לך ניחושי פריסה טובים עבורם. מצב זה מתעלם מבחירות פריסה קודמות.
- `local_mode`: דגל זה קובע היכן רץ מעבר ה-`AIRouting`. אם `False`, ה-`AIRouting` רץ מרחוק דרך שירות Qiskit Transpiler Service. אם `True`, החבילה מנסה להריץ את המעבר בסביבה המקומית שלך עם נפילה חזרה למצב ענן אם התלויות הנדרשות לא נמצאות.

## מעברי סינתזת מעגלים AI
מעברי סינתזת המעגלים של ה-AI מאפשרים לך לאטב חתיכות של סוגי מעגלים שונים ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) על ידי סינתוז מחדש שלהם. דרך טיפוסית להשתמש במעבר הסינתזה היא כך: